<a href="https://colab.research.google.com/github/Anannya-Vyas/machine-learning-projects/blob/main/flower_flask_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1️⃣ Install Flask + Cloudflared
!pip install flask
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
# 2️⃣ Train & save model
import pickle
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

iris = load_iris()
X = iris.data
y = iris.target

model = LogisticRegression(max_iter=200)
model.fit(X, y)

pickle.dump(model, open("model.pkl", "wb"))
print("Model trained and saved successfully!")

In [ ]:
# 3️⃣ Flask app (run in background)
from flask import Flask, request, render_template_string
import threading
import pickle
from sklearn.datasets import load_iris

iris = load_iris()
app = Flask(__name__)
model = pickle.load(open("model.pkl", "rb"))

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Flower Prediction</title>
</head>
<body>
    <h2>🌸 Flower Prediction App 🌸</h2>
    <form action="/predict" method="post">
        Sepal Length: <input type="text" name="sl"><br><br>
        Sepal Width: <input type="text" name="sw"><br><br>
        Petal Length: <input type="text" name="pl"><br><br>
        Petal Width: <input type="text" name="pw"><br><br>
        <input type="submit" value="Predict">
    </form>
    <h3>{{ prediction_text }}</h3>
</body>
</html>
"""

@app.route("/")
def home():
    return render_template_string(HTML)

@app.route("/predict", methods=["POST"])
def predict():
    try:
        features = [float(request.form["sl"]),
                    float(request.form["sw"]),
                    float(request.form["pl"]),
                    float(request.form["pw"])]

        # Simple validation for realistic flower measurements
        if not (0 < features[0] < 10 and 0 < features[1] < 10 and 0 < features[2] < 10 and 0 < features[3] < 10):
            return render_template_string(HTML, prediction_text="❌ Enter realistic values!")

        prediction = model.predict([features])
        flower = iris.target_names[prediction][0]
        return render_template_string(HTML, prediction_text="🌸 Predicted Flower: " + flower)
    except:
        return render_template_string(HTML, prediction_text="❌ Invalid input!")

# Run Flask in background thread
threading.Thread(target=lambda: app.run(host="0.0.0.0", port=5000)).start()

In [ ]:
# 4️⃣ Start Cloudflared tunnel (new cell)
!./cloudflared-linux-amd64 tunnel --url http://localhost:5000